# 2.2 Categorieën vergelijken

**De centrale vraag: hoe verschillen deze groepen van elkaar?**

De ene helft van dit notebook draait om de gereedschapskist — staafdiagrammen, gegroepeerde staven (*grouped bars*), heatmaps en de ontwerpbeslissingen die ze leesbaar maken. 

De andere helft laat twee manieren zien waarop een groepsvergelijking lezers gruwelijk kan misleiden, zelfs wanneer **elk getal op zichzelf 100% klopt**. 

Een derde valkuil — eentje die zo belangrijk is dat hij een heel eigen notebook krijgt — volgt hierna: [02.3-simpsons_paradox](02.3-simpsons_paradox.ipynb).

In [ ]:
# Paragraaf 2.2: Benodigde bibliotheken en visualisatie-gereedschappen importeren
import pandas as pd
import seaborn as sns
from goad_toolkit.visualizer import (
    BasePlot,           # De basisklasse voor al onze visualisaties
    GroupedBarPlot,     # Kant-en-klare gegroepeerde staafgrafiek (voor 2 categorische variabelen)
    HeatmapPlot,        # Heatmap (wanneer minstens één variabele veel categorieën heeft)
    HighlightCategory,  # Laag die één specifieke categorie een accentkleur geeft (grey first)
    PlotSettings,       # Huisstijl-instellingen (figsize, labels, grid, titels)
)

from wa_analyzer.data import load_showcase  # Laadt de meegeleverde showcase-datasets in

## 2.2.1 Een `BarPlot` om te bewaren

In [02.1-baseplot-101](02.1-baseplot-101.ipynb) zag je het skelet van een `BasePlot`-subklasse op een speelgoeddataset. Hier zijn exact dezelfde vijf regels code nog een keer — maar ditmaal is het de klasse die we in de rest van dit hele notebook daadwerkelijk gaan gebruiken.

In [ ]:
# Paragraaf 2.2.1: De BarPlot-klasse (exact 5 regels code)
class BarPlot(BasePlot):
    """Een staafdiagram. Alle styling en opmaak leeft in PlotSettings."""

    def build(self, data: pd.DataFrame, x: str, y: str, **kwargs):
        # Teken direct op self.ax met Seaborn:
        sns.barplot(data=data, x=x, y=y, ax=self.ax, **kwargs)
        # Retourneer altijd fig en ax zodat plot_on-lagen erop kunnen doorkoppelen:
        return self.fig, self.ax

## 2.2.2 De showcase-dataset: pinguïns

Drie soorten, twee geslachten, drie eilanden, 344 vogels. Klein genoeg om handmatig te controleren, en verschillend genoeg dat een slechte grafiek de verschillen niet kan verbergen.

In [ ]:
# Paragraaf 2.2.2: Showcase-dataset pinguïns inladen en inspecteren
penguins = load_showcase("penguins")
penguins.head()  # Inspecteer de eerste 5 rijen: 3 soorten, 3 eilanden, 2 geslachten, 4 metingen

### Waarom je altijd eerst naar `.head()` kijkt voordat je gaat plotten

Niet uit blinde gewoonte — maar om drie cruciale vragen te beantwoorden die een plot-aanroep stilzwijgend fout interpreteert als je niet oplet:
1. **Wat stelt één rij voor?** Hier: één individuele vogel.
2. **Wat voor type data bevat elke kolom?** Is `species` echt een categorische variabele met drie vaste soorten? Is `body_mass_g` echt een getal en niet per ongeluk een stuk tekst met meeteenheden erachter geplakt?
3. **Ontbreekt er opvallende data?** Staat er ergens een `NaN` bij geslacht (`sex`), of een eilandnaam met een typefout?

`.head()` is de snelste en goedkoopste plek om te ontdekken of een kolom iets anders betekent dan de naam doet vermoeden, nog vóórdat je een hele visualisatie bouwt op een verkeerde aanname.

In deze dataset zien we: drie soorten, drie eilanden, twee geslachten en vier continue metingen in millimeters en grammen. Dat maakt `species` (soort) de meest logische eerste variabele om hieronder op te groeperen.

In [ ]:
# Paragraaf 2.2.2: Eenvoudig staafdiagram (nog ongesorteerd)
# 1. Stel de huisstijl in via PlotSettings:
settings = PlotSettings(
    figsize=(8, 4),
    title="Body mass by species",
    xlabel="species",
    ylabel="body mass (g)",
)
# 2. Teken de grafiek:
fig, ax = BarPlot(settings).plot(data=penguins, x="species", y="body_mass_g")

> **Probeer het zelf:** Pas `xlabel` en `ylabel` hierboven aan — bijvoorbeeld naar `"pinguïnsoort"` en `"gemiddeld lichaamsgewicht (gram)"` — en draai de cel opnieuw. 
> Verder hoeft er helemaal niets te veranderen; dit is precies de kracht van het scheiden van opmaak in `PlotSettings`. Kies labels die een lezer echt wil zien, in plaats van alleen maar de ruwe kolomnamen uit de database over te nemen.

### Volgorde is een bewuste beslissing

Standaard tekent Seaborn de categorieën gewoon in de volgorde waarin hij ze tegenkomt in de data. Maar alfabetische volgorde of volgorde van binnenkomst is vrijwel nooit de volgorde die jouw lezer nodig heeft!

**Sorteer altijd op waarde, tenzij de categorie een eigen natuurlijke volgorde heeft** — zoals dagen van de week, leeftijdsklassen of maanden van het jaar. 

Een ongesorteerd staafdiagram dwingt de lezer om zelf in zijn hoofd de vergelijking te gaan maken die jij als data-analist voor hem had moeten doen.

In [ ]:
# Paragraaf 2.2.2: Volgorde is een beslissing (sorteren op gemiddelde massa)
# Bereken het gemiddelde gewicht per soort en sorteer van zwaar naar licht:
order = penguins.groupby("species").body_mass_g.mean().sort_values(ascending=False).index

# Teken de grafiek opnieuw met de expliciete 'order':
fig, ax = BarPlot(settings).plot(data=penguins, x="species", y="body_mass_g", order=order)

### Eerst grijs, dan één kleur (*Grey First*)

Dit is de vijfde van de vijf ontwerprichtlijnen van Jonathan Schwabish. Teken alles in een rustige grijstint, en geef daarna alleen dátgene kleur waar je verhaal over gaat. 

Kleur op deze manier gebruikt fungeert als een laserpointer die direct de aandacht stuurt. Kleur die overal lukraak wordt toegepast is slechts decoratieve ruis: het kost de lezer energie en levert inhoudelijk niets op.

`HighlightCategory` is dit idee uitgewerkt als een herbruikbare laag (`plot_on`) uit Les 2.1: hij kleurt simpelweg de staven opnieuw die al op de as staan. Daardoor werkt hij over jouw `BarPlot`, over een kale `sns.barplot`, of over wat dan ook dat staven heeft getekend. Welke categorie moet oplichten stel je in via `PlotSettings(highlight=[...])`, samen met de rest van de opmaak.

We schrijven dit bewust op als drie losse regels in plaats van één lange keten: definieer de instellingen, teken de basisplot, en leg daarna de laag eroverheen. Elke regel doet één duidelijke taak en is op zichzelf leesbaar.

In [ ]:
# Paragraaf 2.2.2: Grey First (alles grijs, accent op Gentoo)
highlight = "Gentoo"

# We geven de titel een duidelijke conclusie (headline) en stellen het highlight-doel in:
settings = PlotSettings(
    figsize=(8, 4),
    title=f"{highlight} are the heavy ones",
    xlabel="species",
    ylabel="body mass (g)",
    highlight=[highlight],  # Gentoo moet opvallen
)

# 1. Teken de basisstaven (standaard grijs):
bars = BarPlot(settings)
bars.plot(data=penguins, x="species", y="body_mass_g", order=order)

# 2. Leg de HighlightCategory-laag eroverheen (kleurt Gentoo):
bars.plot_on(HighlightCategory(settings))

De bovenstaande `BarPlot` is geen toeval: `goad_toolkit` levert een identieke klasse mee, samen met `GroupedBarPlot`, `HeatmapPlot` en `BarbellPlot`. 

Het zelf uitschrijven hier is een leerzame oefening, net zoals `RegexFeature` in Les 1 dat was: een `BasePlot`-subklasse bestaat uit slechts vijf regels. En als je die vijf regels eenmaal zelf hebt geschreven, zijn de ingebouwde bibliotheekklassen geen zwarte dozen meer. Je begrijpt hoe ze werken, kunt beargumenteren waarom standaardinstellingen zo zijn gekozen, en kunt ze uitbreiden wanneer een standaardplot niet past.

Vanaf Les 3 importeert dit notebook deze klassen direct uit de toolkit in plaats van ze steeds opnieuw te definiëren.

### Twee categorische variabelen tegelijk vergelijken

Kies voor een **gegroepeerd staafdiagram (*grouped bar*)** wanneer beide variabelen weinig niveaus hebben; kies voor een **heatmap** wanneer minstens één van de twee veel niveaus heeft. Beide klassen zitten standaard in de toolkit (`GroupedBarPlot` en `HeatmapPlot`), dus je hoeft hier zelf geen `build()` voor te schrijven.

Hieronder zetten we ze naast elkaar — en dat is de andere helft van `BasePlot` uit 2.1:
`create_figure(n_plots=2)` maakt het raster aan en levert de assen op; `plot_on_axes` plaatst elke plot op een specifieke as. Geen `plt.subplots` nodig, en de titels van de panelen komen direct uit `settings.subplot_titles`.

In [ ]:
# Paragraaf 2.2.2: Twee variabelen tegelijk (Grouped Bar vs. Heatmap)
pair = PlotSettings(
    figsize=(13, 4),
    title="Three species, two other variables",
    subplot_titles=["grouped bars: 3 species x 2 sexes", "heatmap: who lives where"],
    xlabel="species",
    ylabel="body mass (g)",
)

host = BarPlot(pair)
# Maak een figuur aan met 2 panelen naast elkaar:
fig, axes = host.create_figure(n_plots=2)

# Paneel 1: Gegroepeerde staven (soort x geslacht)
host.plot_on_axes(GroupedBarPlot(pair), axes[0],
                  data=penguins, x="species", y="body_mass_g", hue="sex", order=order)

# Paneel 2: Heatmap (wie woont op welk eiland?)
host.plot_on_axes(HeatmapPlot(pair), axes[1],
                  data=penguins, index="species", columns="island",
                  values="body_mass_g", aggfunc="size", fmt=".0f", cmap="Blues")

De heatmap beantwoordt direct een vraag die de staafdiagrammen niet kunnen beantwoorden: **Gentoo en Chinstrap leven elk op exact één specifiek eiland, terwijl Adelie op alle drie de eilanden voorkomt.** 

Dat is een fundamenteel feit over de *structuur van de data*, en het is precies het soort inzicht dat je moet weten vóórdat je conclusies trekt. Een vergelijking tussen soorten en eilanden is immers deels een vergelijking van de eilanden zelf!

### Deze klasse verhuist naar `scripts/plots.py`

Hier geldt exact dezelfde logica als bij `ParseIRCLines` in Les 1: een klasse die alleen in een notebook-cel staat, kan nergens anders worden geïmporteerd. En een dashboard is precies zo'n plek die herbruikbare code nodig heeft! 

De `BarPlot` hierboven is identiek aan wat in `goad_toolkit.visualizer` zit — het zelf schrijven was de oefening. Daarom verhuist hij ongewijzigd naar `scripts/plots.py`. Het dashboard in Les 7 importeert hem rechtstreeks vanuit dat bestand en toont hem in Streamlit zonder ook maar één regel code aan te passen. Omdat `plot()` een gewone Matplotlib-figuur oplevert, kan `st.pyplot()` hem direct weergeven.

In [ ]:
# Paragraaf 2.2.2: Importeren vanuit scripts/plots.py
from scripts.plots import BarPlot as ImportedBarPlot

# Exact dezelfde klasse, maar nu geïmporteerd als herbruikbare module:
fig, ax = ImportedBarPlot(settings).plot(
    data=penguins, x="species", y="body_mass_g", order=order, color="#cccccc"
)

## 2.2.3 Twee manieren waarop een groepsvergelijking misleidt

Dezelfde getallen, een andere grafiek, en een compleet verkeerde indruk. De showcase voor deze tweede helft is de passagierslijst van de **Titanic**: één rij per passagier, of ze het overleefd hebben (`survived`), in welke klasse ze zaten, op welk dek (`deck`) hun hut lag en in welke haven ze aan boord gingen.

In [ ]:
# Paragraaf 2.2.3: De Titanic-dataset inladen
titanic = load_showcase("titanic")
titanic.head()  # Bekijk kolommen: pclass, survived, deck, age, etc.

891 passagiers, 15 kolommen. Zowel `deck` als `age` bevatten aanzienlijke gaten (ontbrekende waarden), en dat is cruciaal voor wat nu volgt. 

Twee vragen, één dataset:
1. **Welke combinatie van dek en klasse vervoerde de meeste passagiers?** — Een gewone telvraag, die zo meteen helemaal de mist in gaat om een reden die niets met tellen te maken heeft.
2. **Welk dek had het hoogste overlevingspercentage?** — Een percentage (*proportie*), die zo meteen misgaat om de oudste reden waarom percentages misleiden.

### Valkuil 1: Te veel staven, zonder enige ordening (Regenboog-chaos)

In [ ]:
# Paragraaf 2.2.3: Valkuil 1 — 21 ongeordende regenboogstaven vs. één scherpe vraag
# Vul ontbrekende dekken in met 'unknown' zodat derdeklasse-passagiers niet stilzwijgend verdwijnen:
by_deck = titanic.assign(deck=titanic.deck.astype(object).fillna("unknown"))
by_deck = by_deck.groupby(["deck", "class"], observed=True).size().reset_index(name="n")  # ty: ignore[no-matching-overload]

decks = PlotSettings(
    figsize=(13, 4),
    title="Same passengers, two ways of asking",
    subplot_titles=["unordered, rainbow, 14 bars", "one question, ordered, one colour"],
    xlabel="passengers",
    ylabel="",
)

host = BarPlot(decks)
fig, axes = host.create_figure(n_plots=2)

# Links: Onleesbare regenboog-chaos zonder ordening
host.plot_on_axes(GroupedBarPlot(decks), axes[0],
                  data=by_deck, x="n", y="deck", hue="class", palette="rainbow")

# Rechts: Eén heldere vraag, gesorteerd van veel naar weinig, in rustig grijs
top = titanic.groupby("class", observed=True).size().sort_values(ascending=False)  # ty: ignore[no-matching-overload]
host.plot_on_axes(BarPlot(decks), axes[1],
                  data=top.reset_index(name="n"), x="n", y="class", color="#cccccc")

Let goed op het dek met het label **`unknown`** (onbekend)! 

Een standaard `groupby(["deck", "class"])` zou **stilzwijgend elke rij weggooien** waar `deck` ontbreekt (`NaN`). En voor de derde klasse geldt dat voor maar liefst **479 van de 491 passagiers**: in het ruim (*steerage*) werden hutnummers vrijwel nooit geregistreerd! Derdeklasse-passagiers bestaan dus nauwelijks in een grafiek op basis van dek, tenzij je expliciet aangeeft wat je met ontbrekende waarden doet.

Dat is ook meteen het antwoord op de vraag *"waar is de rest van de derde klasse gebleven?"* in het rechterpaneel: die zat vanaf het begin al niet in het linkerpaneel, omdat ze simpelweg ontbraken. 

Het gat opvullen als een eigen categorie (`unknown`), in plaats van rijen geruisloos weggooien, is wat de ontwerprichtlijn **"show the data"** werkelijk vereist. Het alternatief is een misleidende grafiek die stilletjes tweederde van het schip weglaat en dat doodleuk een 'vergelijking' noemt.

De linkergrafiek probeert de vraag *"welke dek-klasse combinatie had de meeste passagiers"* te beantwoorden met maar liefst **21 staven in 21 regenboogkleuren zonder enige ordening**. Technisch gezien klopt elk getal, maar in de praktijk is het volstrekt onleesbaar.

De rechtergrafiek beantwoordt een veel scherpere, helderdere vraag — *welke klasse vervoerde de meeste passagiers?* — en die lees je in één enkele oogopslag af. De linkergrafiek is niet rekenkundig fout, maar faalt compleet in het communiceren van een duidelijke boodschap.

---

### Valkuil 2: Een percentage zónder zijn noemer (*A proportion without its denominator*)

Nu de tweede vraag: welk dek had het beste overlevingspercentage? Kijk wat er gebeurt als je een percentage plot zonder de absolute aantallen te tonen waarop dat percentage rust!

In [ ]:
# Paragraaf 2.2.3: Valkuil 2 — Een percentage zonder zijn noemer
# Bereken overlevingspercentage (mean) én aantal passagiers (size) per dek:
rate = titanic.groupby("deck", observed=True).survived.agg(["mean", "size"])
rate = rate.sort_values("mean", ascending=False).reset_index()
rate["pct"] = rate["mean"] * 100

proportion = PlotSettings(
    figsize=(13, 4),
    title="A rate, and what it rests on",
    subplot_titles=["survival rate by deck (%)", "...how many people that is"],
    xlabel="",
    ylabel="deck",
)

host = BarPlot(proportion)
fig, axes = host.create_figure(n_plots=2)

# Links: Het percentage (Dek G lijkt spectaculair hoog!)
host.plot_on_axes(BarPlot(proportion), axes[0],
                  data=rate, x="pct", y="deck", color="#cccccc")

# Rechts: Het aantal observaties (Dek G had in werkelijkheid maar 4 passagiers!)
host.plot_on_axes(BarPlot(proportion), axes[1],
                  data=rate, x="size", y="deck", color="#cccccc")

Kijk eens naar **Dek G**: dat heeft een overlevingspercentage waar je een spectaculaire krantenkop over zou kunnen schrijven... maar er zaten welgeteld **slechts 4 mensen** op dat dek!

**Toon of annoteer ALTIJD het absolute aantal (de noemer) achter elk percentage dat je plot.** 

Een percentage over een handjevol waarnemingen is statistisch gezien pure ruis en toeval. Een staafdiagram geeft zo'n percentage echter exact hetzelfde visuele gewicht als een percentage dat gebaseerd is op honderden mensen. Dat is precies wat het linkerpaneel hierboven doet — en exact wat het rechterpaneel direct ontmaskert!

> **Stel jezelf deze vraag elke keer dat je een percentage plot:** Op hoeveel werkelijke waarnemingen (observaties) rust dit getal eigenlijk?

---

**Hoe we nu verder gaan:** Deze twee valkuilen hebben één ding gemeen: het zijn samenvattingen die cruciale informatie verbergen die je mening compleet zou hebben veranderd. 

Er is echter nog een derde, nog gevaarlijkere valkuil. En die heeft geen slordige grafiek nodig om op te duiken: hij treedt op wanneer de getallen zélf compleet van richting omdraaien afhankelijk van hoe je groepeert, terwijl elke tussenstap wiskundig 100% klopt! 

Dat fenomeen heet **Simpson's Paradox** en krijgt een heel eigen notebook: [02.3-simpsons_paradox](02.3-simpsons_paradox.ipynb).